# Man-in-the-Middle — Owner-Only Spoofing

A single GCS both owns and monitors this vehicle. An attacker has man-in-the-
middled the link and, once the drone reaches a chosen waypoint
(`MISSION_CURRENT.seq >= trigger_seq`), injects one fabricated
`GLOBAL_POSITION_INT` — spoofed to look like it came from the vehicle itself
— toward that GCS.

This is `spoof_owner_gcs`: the same attack as `spoof_gcs`
(see `13-mitm_spoof_gcs.ipynb`) but hardcoded to always target index 0, the
owner — no `target_mask` needed, and it still does the right thing even with
only one GCS in the picture (the common case: an operator flying their own
drone with no second observer to compare against).

The real telemetry keeps flowing too, so the operator's map briefly shows a
bogus point far from the real mission path, planted alongside the genuine
track rather than replacing it.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.runtime.mitm import SpoofOwnerGCSStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0  # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 ← attacker spoofs the owner GCS here
home_wp = ENU(x=0, y=0, z=0)
north_100 = ENU(x=0, y=100, z=cruise_alt)
north_200 = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]

# Fabricated position reported to the operator: 150 m SOUTH of origin, well
# off the real (northbound) mission path so the spoof is unmistakable.
spoof_target = gra_origin.unpose().to_abs(ENU(x=0, y=-150, z=cruise_alt))
print(f"Spoof target: lat={spoof_target.lat:.7f}, lon={spoof_target.lon:.7f}")

## Vehicle

In [ ]:
mission_path = "simulator/planner/missions/mitm_north.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=mission_path,
    navigation_speed=speed,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcss=[SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}", record_positions=True)],
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + MITM owner-only spoof

In [ ]:
orac = Oracle()

orac.add_vehicle(vehicle)

# No target_mask param: SpoofOwnerGCSStrategy always targets index 0 (the owner).
vehicle.mitm = SpoofOwnerGCSStrategy(
    trigger_seq=3,
    spoof_lat=spoof_target.lat,
    spoof_lon=spoof_target.lon,
    spoof_alt=cruise_alt,
)

simulator = Simulator(oracle=orac, visualizer=gaz, verbose=1)

simulator.preview()

In [ ]:
simulator.run()

## What to observe

- `simulator/logs/mitm/mitm_1.log` — `MITM spoof: reporting fake position ...
  to GCS [0]`.
- The drone itself flies its real north mission throughout — this attack
  never touches Logic or the flight controller, only what the GCS is told.
- The GCS's recorded trajectory shows the real north track plus one bogus
  point far to the south.

In [ ]:
orac.plot_trajectories(oracle=False, gcss="all");